# 02 · Iterators & Generators

**Module 09 – Advanced Functions** · Estimated time: 55 min

> Expands 📗 *Iterators and Generators* with the iterator protocol, custom iterators, `yield from`, generator pipelines, `send()`, and infinite generators.

### 🎯 What you will learn
- Iterable vs iterator; how a `for` loop really works
- `iter()` and `next()`, `StopIteration`
- Writing a custom iterator class (`__iter__`, `__next__`)
- Generator functions with `yield` (📗 cubes, Fibonacci)
- Why generators save memory (lazy evaluation)
- Generator expressions
- `yield from`, infinite generators, data pipelines
- `send()` and two-way generators (advanced)

## 1. Iterable vs iterator

- An **iterable** is anything you can loop over (list, str, dict, file, range...). It has `__iter__()`.
- An **iterator** is the object that produces values **one at a time**. It has `__next__()` and remembers its position.

`iter(iterable)` gives you an iterator; `next(iterator)` gets the next value.

In [1]:
s = "hello"
for letter in s:          # a string is iterable
    print(letter, end=" ")
print()

h e l l o 


In [2]:
next(s)                   # TypeError: 'str' object is not an iterator

TypeError: 'str' object is not an iterator

In [3]:
s_iter = iter(s)          # get an iterator
print(next(s_iter), next(s_iter), next(s_iter))
print(list(s_iter))       # the rest: iterators remember where they are

h e l
['l', 'o']


### What a `for` loop actually does

In [4]:
items = [10, 20, 30]
it = iter(items)
while True:
    try:
        item = next(it)
    except StopIteration:      # signals "no more values"
        break
    print(item)

10
20
30


❗ An iterator can only be consumed **once**:

In [5]:
it = iter([1, 2, 3])
print(list(it), list(it))     # second time: empty

[1, 2, 3] []


## 2. A custom iterator class

In [6]:
class Countdown:
    def __init__(self, start):
        self.current = start

    def __iter__(self):
        return self

    def __next__(self):
        if self.current <= 0:
            raise StopIteration
        value = self.current
        self.current -= 1
        return value

for n in Countdown(5):
    print(n, end=" ")

5 4 3 2 1 

That's a lot of boilerplate. Generators do the same with far less code.

## 3. Generator functions: `yield` (📗)

A function containing `yield` becomes a **generator function**. Calling it returns a **generator** (an iterator) without running the body. Each `next()` runs until the next `yield`, hands out that value, and **pauses**, keeping all local variables.

In [7]:
def countdown(start):
    while start > 0:
        yield start
        start -= 1

print(list(countdown(5)))

[5, 4, 3, 2, 1]


In [8]:
def gencubes(n):
    """Generator for the cube of numbers."""
    for num in range(n):
        yield num ** 3

for x in gencubes(10):
    print(x, end=" ")

0 1 8 27 64 125 216 343 512 729 

### Seeing the pause/resume

In [9]:
def simple_gen():
    print("  start")
    yield 1
    print("  resumed after 1")
    yield 2
    print("  resumed after 2, finishing")

g = simple_gen()
print(g)
print(next(g))
print(next(g))
try:
    next(g)
except StopIteration:
    print("StopIteration: generator is exhausted")

<generator object simple_gen at 0x10835c940>
  start
1
  resumed after 1
2
  resumed after 2, finishing
StopIteration: generator is exhausted


## 4. Why generators? Memory efficiency (📗)

In [10]:
def genfibon(n):
    """Generate a Fibonacci sequence up to n terms, lazily."""
    a, b = 1, 1
    for _ in range(n):
        yield a
        a, b = b, a + b

def fibon(n):
    """Build the whole list in memory."""
    a, b = 1, 1
    output = []
    for _ in range(n):
        output.append(a)
        a, b = b, a + b
    return output

print(list(genfibon(10)))
print(fibon(10))

[1, 1, 2, 3, 5, 8, 13, 21, 34, 55]
[1, 1, 2, 3, 5, 8, 13, 21, 34, 55]


In [11]:
import sys
print("list of 1,000,000 squares:", sys.getsizeof([i * i for i in range(1_000_000)]), "bytes")
print("generator of 1,000,000 squares:", sys.getsizeof((i * i for i in range(1_000_000))), "bytes")

list of 1,000,000 squares: 8448728 bytes
generator of 1,000,000 squares: 216 bytes


The generator is tiny no matter how many items, because values are computed **on demand**. This is essential for huge files, streams and infinite sequences. 📗 *"If the output has the potential of taking up a large amount of memory and you only intend to iterate through it, use a generator."*

## 5. Generator expressions

In [12]:
squares = (x * x for x in range(5))     # parentheses → generator
print(next(squares), list(squares))
print(sum(x * x for x in range(10)))     # no extra parentheses needed as a sole argument
print(max(len(w) for w in ["apple", "kiwi", "banana"]))

0 [1, 4, 9, 16]
285
6


## 6. Infinite generators

In [13]:
def natural_numbers():
    n = 1
    while True:            # never ends, which is fine because values are produced lazily
        yield n
        n += 1

import itertools
print(list(itertools.islice(natural_numbers(), 10)))    # take the first 10

def id_generator(prefix):
    for n in itertools.count(1):
        yield f"{prefix}-{n:04d}"

ids = id_generator("ORD")
print(next(ids), next(ids), next(ids))

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
ORD-0001 ORD-0002 ORD-0003


## 7. `yield from`: delegate to another iterable

In [14]:
def flatten(nested):
    for item in nested:
        if isinstance(item, list):
            yield from flatten(item)     # yield each value from the sub-generator
        else:
            yield item

print(list(flatten([1, [2, [3, 4]], 5, [[6]]])))

def chain(*iterables):
    for it in iterables:
        yield from it

print(list(chain("ab", [1, 2], range(3))))

[1, 2, 3, 4, 5, 6]
['a', 'b', 1, 2, 0, 1, 2]


## 8. Generator pipelines: process data like an assembly line

Each stage is lazy, so even a multi-GB log file is processed with almost no memory:

In [15]:
log_lines = [
    "2026-10-20 INFO user=asha action=login",
    "2026-10-20 ERROR user=ravi action=payment failed",
    "2026-10-20 INFO user=meena action=logout",
    "2026-10-21 ERROR user=asha action=upload timeout",
]

def read_lines(lines):                    # in real life: for line in open("app.log")
    for line in lines:
        yield line.strip()

def only_errors(lines):
    for line in lines:
        if " ERROR " in line:
            yield line

def extract_user(lines):
    for line in lines:
        yield line.split("user=")[1].split()[0]

pipeline = extract_user(only_errors(read_lines(log_lines)))
print(list(pipeline))

['ravi', 'asha']


## 9. Two-way generators: `send()` (advanced)

`yield` is also an **expression** that can *receive* a value sent in with `gen.send(value)`:

In [16]:
def running_average():
    total, count, average = 0, 0, None
    while True:
        value = yield average        # hand out the average, wait for the next value
        total += value
        count += 1
        average = total / count

avg = running_average()
next(avg)                            # "prime" the generator (run up to the first yield)
for v in [10, 20, 30, 40]:
    print(avg.send(v))

10.0
15.0
20.0
25.0


(This mechanism is the historical basis of `async`/`await`, Module 13.)

## 10. Generator gotchas
- Generators are **single-use**; re-create them to iterate again.
- `len()` and indexing don't work on generators. Convert with `list()` if you need them (memory cost!).
- A `return value` in a generator ends it; the value is attached to `StopIteration.value`.

## 📝 Summary
- Iterable → `iter()` → iterator → `next()` until `StopIteration`.
- `yield` turns a function into a generator: lazy, pausable, memory-efficient.
- Generator expressions `( ... )` are the lazy cousins of list comprehensions.
- `yield from` delegates; generators compose into pipelines; `send()` enables coroutines.

## 🏋️ Practice (📗 Iterators and Generators Homework)
1. Create a generator `gensquares(N)` that generates the squares of numbers up to N.
2. Create a generator `rand_num(low, high, n)` that yields `n` random numbers in the range.
3. Use `iter()` to convert the string `s = 'hello'` into an iterator and print its first letter with `next()`.
4. Explain a use case for a generator using `yield` instead of a normal function with `return`.
5. **Extra credit:** can you explain what `gencomp` is in:
   ```python
   my_list = [1, 2, 3, 4, 5]
   gencomp = (item for item in my_list if item > 3)
   for item in gencomp:
       print(item)
   ```
6. Write `chunks(lst, size)` that yields successive chunks: `chunks([1,2,3,4,5], 2)` → `[1,2] [3,4] [5]`.

<details><summary>Show solutions</summary>

```python
import random

def gensquares(N):
    for i in range(N):
        yield i ** 2

def rand_num(low, high, n):
    for _ in range(n):
        yield random.randint(low, high)

s = 'hello'
s = iter(s)
print(next(s))

# 4: when the output may be large or infinite and you only iterate over it once:
#    reading a huge file line by line, streaming API results, generating IDs.

# 5: gencomp is a generator expression; it lazily yields 4 and 5.

def chunks(lst, size):
    for i in range(0, len(lst), size):
        yield lst[i:i + size]
# (Python 3.12+: itertools.batched(lst, size) does this, yielding tuples)
```
</details>